In [1]:
from dlfs.base import Module, Loss, Optimizer, Layer

from dlfs.modules import SequentialWrapper
from dlfs.layers import DenseLayer, ConvolutionalLayer
from dlfs.activation import ReLU, Sigmoid

from dlfs.loss import MSE_Loss, BCE_Loss
from dlfs.optimizers import Optimizer_Adam
from dlfs.helpers import dilate, pad_to_shape

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

In [2]:
class ConvTransposeLayer(Layer):

    def __init__(self, input_channels: tuple, output_channels: int, kernel_size: int, stride: int = 1, padding: int = 0, output_padding=0) -> None:

        self.input_channels = input_channels
        self.output_channels = output_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.output_padding = output_padding

        # Create output and kernel shapes
        self.kernel_size = kernel_size

        # Initialize layer parameters
        self.kernels = np.random.randn(output_channels, input_channels, kernel_size, kernel_size)
        self.biases = np.random.randn(output_channels)

    def forward(self, inputs: np.ndarray, training=False) -> None:

        if isinstance(self.output_padding, tuple):
            output_padding_H, output_padding_W = self.output_padding
        else:
            output_padding_H = output_padding_W = self.output_padding

        n_samples, _, H, W = inputs.shape

        # Compute output shape
        H_out = (H - 1) * self.stride - 2 * self.padding + self.kernel_size + output_padding_H
        W_out = (W - 1) * self.stride - 2 * self.padding + self.kernel_size + output_padding_W

        # Store inputs for later use
        self.inputs = inputs

        # Output is 4D tensor of shape (n_samples, output_channels, height, width)
        self.output = np.zeros((n_samples, self.output_channels, H_out, W_out))

        # Add bias to output
        self.output += self.biases[None, :, None, None]

        # Loop through each sample, output channel and input channel
        for i in range(n_samples):
            for j in range(self.output_channels):
                for k in range(self.input_channels):

                    current_input = inputs[i, k]

                    current_input = dilate(current_input, self.stride)

                    current_input = np.pad(current_input, pad_width=self.padding)
                    
                    conv_out = signal.convolve2d(current_input, self.kernels[j, k], mode='full')

                    H_conv, W_conv = conv_out.shape

                    # crop center if conv_out bigger than expected
                    start_h = (H_conv - H_out) // 2
                    start_w = (W_conv - W_out) // 2
                    conv_out = conv_out[start_h:start_h+H_out, start_w:start_w+W_out]

                    self.output[i, j] += conv_out

    def backward(self, delta: np.ndarray) -> None:
        """
        Backward pass using the convolutional layer. Creates gradient attributes with respect to kernels, biases and inputs.

        Parameters
        ----------
        delta : np.ndarray
            Accumulated gradient obtained by backpropagation.

        Returns
        -------
        None
        """
        # Initialize gradient attributes
        self.dkernels = np.zeros(self.kernels.shape)
        self.dbiases = np.sum(delta, axis=(0,2,3))
        self.dinputs = np.zeros(self.inputs.shape)

        # Number of samples, first dimension
        n_samples = self.inputs.shape[0]

        # Loop through each sample, output channel and input channel
        for i in range(n_samples):

            for j in range(self.output_channels):
                for k in range(self.input_channels):

                    current_input = self.inputs[i, k]

                    dkernels = self._calculate_kernel_gradient_transpose(current_input, 
                                                                         delta[i, j], 
                                                                         self.kernels[j, k], 
                                                                         stride=self.stride,
                                                                         padding=self.padding)
                    dinputs = self._calculate_input_gradient_transpose(current_input, 
                                                                       delta[i, j], 
                                                                       self.kernels[j, k], 
                                                                       stride=self.stride,
                                                                       padding=self.padding)

                    self.dkernels[j, k] += dkernels
                    self.dinputs[i, k] += dinputs

    def get_parameters(self):
        param_names = ["kernels", "biases"]
        return super()._filter_parameters(param_names)

    def _calculate_kernel_gradient(self, inputs: np.ndarray, delta: np.ndarray, kernel: np.ndarray, stride: int = 1) -> np.ndarray:
        """
        Helper method for calculating kernel gradient.

        Parameters
        ----------
        inputs : np.ndarray
            Current sample the gradient is calculated for.

        delta : np.ndarray
            Accumulated gradient obtained by backpropagation.

        kernel : np.ndarray
            Kernel used in convolutional layer.

        stride : int, default=1
            Step size at which the kernel moves across the input.

        Returns
        -------
        kernel_grad : np.ndarray
            Kernel gradient.
        """

        if stride > 1:

            # If stride is present delta needs to be dilated
            delta_dilated = dilate(delta, stride)

            delta_dilated_height, delta_dilated_width = delta_dilated.shape[-2:]
            input_height, input_width = inputs.shape[-2:]
            kernel_shape = kernel.shape[-1]

            if delta_dilated_height == input_height - kernel_shape + 1 and delta_dilated_width == input_width - kernel_shape + 1:
                # If dilated delta shape matches the needed correlation shape gradient can be computed
                dkernel = signal.correlate2d(inputs, delta_dilated, "valid")
            else:
                # If dilated delta shape doesn't match the needed correlation shape padding is needed
                new_delta_shape = (input_height - kernel_shape + 1, input_width - kernel_shape + 1)
                delta_dilated_padded = pad_to_shape(delta_dilated, new_delta_shape)
                dkernel = signal.correlate2d(inputs, delta_dilated_padded, "valid")

        else:
            # Gradient with respect to kernel is valid cross correlation between inputs and delta
            dkernel = signal.correlate2d(inputs, delta, "valid")

        return dkernel

    def _calculate_input_gradient(self, inputs: np.ndarray, delta: np.ndarray, kernel: np.ndarray, stride: int = 1):
        """
        Helper method for calculating input gradient.

        Parameters
        ----------
        inputs : np.ndarray
            Current sample the gradient is calculated for.

        delta : np.ndarray
            Accumulated gradient obtained by backpropagation.

        kernel : np.ndarray
            Kernel used in convolutional layer.

        stride : int, default=1
            Step size at which the kernel moves across the input.

        Returns
        -------
        input_grad : np.ndarray
            Input gradient.
        """

        if stride > 1:

            delta_dilated = dilate(delta, stride)

            delta_dilated_height, delta_dilated_width = delta_dilated.shape[-2:]
            input_height, input_width = inputs.shape[-2:]
            kernel_shape = kernel.shape[-1]

            if delta_dilated_height == input_height - kernel_shape + 1 and delta_dilated_width == input_width - kernel_shape + 1:
                # If dilated delta shape matches the needed coonvolution shape gradient can be computed
                dinput = signal.convolve2d(delta_dilated, kernel, "full")
            else:
                # If dilated delta shape doesn't match the needed convolution shape padding is needed
                new_delta_shape = (input_height - kernel_shape + 1, input_width - kernel_shape + 1)
                delta_dilated_padded = pad_to_shape(delta_dilated, new_delta_shape)
                dinput = signal.convolve2d(delta_dilated_padded, kernel, "full")

        else:
            # Gradient with respect to inputs is full convolution between delta and kernel
            dinput = signal.convolve2d(delta, kernel, "full")

        return dinput

    def _calculate_kernel_gradient_transpose(self, inputs: np.ndarray, delta: np.ndarray, kernel: np.ndarray,
                                            stride: int = 1, padding: int = 0) -> np.ndarray:
        """
        Helper for ConvTranspose2d: gradient wrt kernel.

        Parameters
        ----------
        inputs : np.ndarray
            Input feature map (before transposed convolution).
        delta : np.ndarray
            Gradient of loss wrt layer output (grad_output).
        kernel : np.ndarray
            Kernel used in this layer.
        stride : int
            Stride used in forward.
        padding : int
            Padding used in forward.

        Returns
        -------
        dkernel : np.ndarray
            Gradient wrt kernel weights.
        """

        if stride > 1:

            # If stride is present delta needs to be dilated
            inputs_dilated = dilate(inputs, stride)

            inputs_dilated_height, inputs_dilated_width = inputs_dilated.shape[-2:]
            delta_height, delta_width = delta.shape[-2:]
            kernel_shape = kernel.shape[-1]

            if inputs_dilated_height == delta_height - kernel_shape + 1 and inputs_dilated_width == delta_width - kernel_shape + 1:
                # If dilated delta shape matches the needed correlation shape gradient can be computed
                dkernel = signal.correlate2d(inputs_dilated, delta, "valid")
            else:
                # If dilated delta shape doesn't match the needed correlation shape padding is needed
                new_inputs_shape = (delta_height - kernel_shape + 1, delta_width - kernel_shape + 1)
                inputs_dilated_padded = pad_to_shape(inputs_dilated, new_inputs_shape)
                dkernel = signal.correlate2d(inputs_dilated_padded, delta, "valid")

        else:
            # Gradient with respect to kernel is valid cross correlation between inputs and delta
            dkernel = signal.correlate2d(inputs, delta, "valid")

        """        
        # 1. Dilate the input
        input_dilated = dilate(inputs, stride)

        # 2. Pad the dilated input (as in forward)
        if padding > 0:
            input_dilated = np.pad(input_dilated, pad_width=padding)

        # 3. Cross-correlation of input_dilated and grad_output
        dkernel = signal.correlate2d(input_dilated, delta, mode='valid')
        """

        return dkernel

    def _calculate_input_gradient_transpose(self, inputs: np.ndarray, delta: np.ndarray, kernel: np.ndarray,
                                        stride: int = 1, padding: int = 0) -> np.ndarray:
        """
        Helper for ConvTranspose2d: gradient wrt inputs.

        Parameters
        ----------
        inputs : np.ndarray
            Input feature map (before transposed convolution).
        delta : np.ndarray
            Gradient of loss wrt layer output (grad_output).
        kernel : np.ndarray
            Kernel used in this layer.
        stride : int
            Stride used in forward.
        padding : int
            Padding used in forward.

        Returns
        -------
        dinput : np.ndarray
            Gradient wrt layer input (to pass backward).
        """

        # 1. Flip kernel 180 degrees
        kernel_rot = np.rot90(kernel, 2)

        # 2. Full convolution of grad_output with flipped kernel
        dinput = signal.convolve2d(delta, kernel_rot, mode='valid')

        # 3. Remove padding that was added in forward
        if padding > 0:
            dinput = dinput[padding:-padding, padding:-padding]

        # 4. Undilate (reverse the stride)
        if stride > 1:
            dinput = dinput[::stride, ::stride]

        return dinput

In [3]:
class ConvTranspose2D:
    def __init__(self, input_channels, output_channels, kernel_size, stride=1, padding=0):
        self.input_channels = input_channels
        self.output_channels = output_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        # Randomly initialize kernels and biases
        self.kernels = np.random.randn(output_channels, input_channels, kernel_size, kernel_size)
        self.biases = np.random.randn(output_channels)

    def forward(self, inputs):
        """
        Forward pass for transposed convolution.

        Parameters
        ----------
        inputs : np.ndarray
            Input of shape (batch_size, input_channels, H, W)

        Returns
        -------
        np.ndarray
            Output of shape (batch_size, output_channels, H_out, W_out)
        """
        n_samples, _, H, W = inputs.shape
        
        # Compute output shape
        H_out = (H - 1) * self.stride - 2 * self.padding + self.kernel_size
        W_out = (W - 1) * self.stride - 2 * self.padding + self.kernel_size
        
        # Initialize output
        output = np.zeros((n_samples, self.output_channels, H_out, W_out))
        
        # Loop over batch, output channels, input channels
        for i in range(n_samples):
            for j in range(self.output_channels):
                for k in range(self.input_channels):
                    # Upsample input by stride
                    in_up = self.upsample_by_stride(inputs[i, k])
                    
                    # Add padding if specified
                    if self.padding > 0:
                        in_up = np.pad(in_up, self.padding, mode='constant')
                    
                    # Convolve using full mode (spreads input over output)
                    # Either rotate kernel and use correlate2d OR use convolve2d directly
                    output[i, j] += signal.convolve2d(in_up, self.kernels[j, k], mode='valid')
        
        return output

In [4]:
x = np.random.randn(5, 3, 27, 36)

layer1 = ConvolutionalLayer(input_shape=(3, 27, 36), output_channels=6, kernel_size=3, stride=2, padding=1)
layer2 = ConvTransposeLayer(input_channels = 6, output_channels = 3, kernel_size=3, stride=2, padding=3, output_padding=(4, 5))

layer1.forward(x)

print(layer1.output.shape)

layer2.forward(layer1.output)

print(layer2.output.shape)

(5, 6, 14, 18)
(5, 3, 27, 36)


In [5]:
delta = np.random.randn(*layer2.output.shape)

layer2.backward(delta)

ValueError: Target shape must be larger than the array shape.